# KDD Process Project

This notebook implements the **Knowledge Discovery in Databases (KDD)** process.  We focus on selection, preprocessing, transformation, data mining and interpretation using the credit card fraud dataset.  Unlike the previous methodologies, here we experiment with an unsupervised anomaly detection algorithm.

## Selection & Preprocessing

All records are retained to train the unsupervised model.  Features are standardised to zero mean and unit variance.  No class labels are used during training.

### Expert Review & Recommendations
In anomaly detection, retaining as much normal behaviour as possible helps the model learn what ‘normal’ looks like.  Care must be taken not to leak label information.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Load the data
df = pd.read_csv('dataset/creditcard.csv')
X = df.drop(columns=['Class'])
y = df['Class']
# Standardise features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# For unsupervised anomaly detection, we keep all data (no train/test split)

## Transformation, Data Mining & Interpretation

We train an Isolation Forest to learn the ‘normal’ pattern of credit card transactions.  It assigns anomaly scores; points with scores below a threshold are labelled as potential fraud.  Mapping these predictions back to the known labels allows us to evaluate how well the unsupervised approach performs.

### Expert Review & Recommendations
Isolation Forest offers a quick baseline.  More sophisticated methods like autoencoders or one‑class SVMs could improve performance.  In practice, unsupervised methods are valuable when labels are scarce or unreliable.

In [2]:
# Train an Isolation Forest to detect anomalies
from sklearn.ensemble import IsolationForest
iso = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)
iso.fit(X_scaled)
# Predict anomalies (-1 for outliers, 1 for inliers)
scores = iso.decision_function(X_scaled)
y_pred = iso.predict(X_scaled)
# Map predictions to fraud (1) or not (0) where -1 is anomaly
y_pred_labels = np.where(y_pred == -1, 1, 0)
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
cm = confusion_matrix(y, y_pred_labels)
prec = precision_score(y, y_pred_labels)
rec = recall_score(y, y_pred_labels)
f1 = f1_score(y, y_pred_labels)
print('Confusion matrix:', cm)
print(f'Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}')

Confusion matrix: [[284113    202]
 [   409     83]]
Precision: 0.2912, Recall: 0.1687, F1: 0.2136
